## Normal distribution dashboards (all 8 axes)

This section starts again from the raw CSV so it can run on its own. Each dashboard has its own filter panel and an **Apply filter** button. Timestamps in the file are UTC.

### Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import ipywidgets as widgets
from IPython.display import display, clear_output

### Loading the data
All eight axes and the timestamp. The empty Axis #9 to #14 columns and the Trait column are left out.

In [3]:
axes_cols = [f'Axis #{i}' for i in range(1, 9)]

robot_df = pd.read_csv('robot_data.csv', usecols=axes_cols + ['Time'])
print(robot_df.shape)
robot_df.head()

(39672, 9)


,Axis #1,Axis #2,Axis #3,Axis #4,Axis #5,Axis #6,Axis #7,Axis #8,Time
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-10-17T12:18:23.660Z
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-10-17T12:18:25.472Z
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-10-17T12:18:27.348Z
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-10-17T12:18:29.222Z
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-10-17T12:18:31.117Z


### Share of zero readings
About 65% of every axis is zero, so a distribution of all readings is one tall spike at 0. The dashboards default to non-zero readings for that reason.

In [4]:
zero_share = (robot_df[axes_cols] == 0).mean().mul(100).round(1)
print(zero_share)
print('\nRows where every axis is zero:', round((robot_df[axes_cols].sum(axis=1) == 0).mean() * 100, 1), '%')

Axis #1    65.3
Axis #2    65.1
Axis #3    65.1
Axis #4    65.5
Axis #5    65.2
Axis #6    65.7
Axis #7    65.6
Axis #8    65.4
dtype: float64

Rows where every axis is zero: 64.2 %


### Timestamp feature engineering
Time is parsed to a datetime, then broken into date, hour, elapsed minutes and the gap since the previous reading. Day part and elapsed quarter give the portions used for filtering, and an active row is any row where at least one axis is above zero.

In [5]:
robot_df['Time'] = pd.to_datetime(robot_df['Time'], utc=True)
robot_df = robot_df.sort_values('Time').reset_index(drop=True)

robot_df['Date'] = robot_df['Time'].dt.strftime('%Y-%m-%d')
robot_df['Hour'] = robot_df['Time'].dt.hour
robot_df['Elapsed Min'] = (robot_df['Time'] - robot_df['Time'].iloc[0]).dt.total_seconds() / 60
robot_df['Time Gap'] = robot_df['Time'].diff().dt.total_seconds().fillna(0)

robot_df['Day Part'] = pd.cut(
    robot_df['Hour'],
    bins=[-1, 5, 11, 17, 23],
    labels=['Night (00-05)', 'Morning (06-11)', 'Afternoon (12-17)', 'Evening (18-23)']
)
robot_df['Elapsed Quarter'] = pd.qcut(robot_df['Elapsed Min'], 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
robot_df['Active Row'] = robot_df[axes_cols].sum(axis=1) > 0

for col in ['Day Part', 'Elapsed Quarter']:
    robot_df[col] = robot_df[col].astype(str)

print('Readings more than 60 seconds after the previous one:', (robot_df['Time Gap'] > 60).sum())
print(robot_df.groupby('Day Part')['Active Row'].agg(['size', 'mean']).round(3))
robot_df[['Time', 'Date', 'Hour', 'Elapsed Min', 'Time Gap', 'Day Part', 'Elapsed Quarter', 'Active Row']].head()

Readings more than 60 seconds after the previous one: 2
                    size   mean
Day Part                       
Afternoon (12-17)   9779  0.277
Evening (18-23)    10684  0.487
Morning (06-11)     8481  0.441
Night (00-05)      10728  0.236


,Time,Date,Hour,Elapsed Min,Time Gap,Day Part,Elapsed Quarter,Active Row
0,2022-10-17 12:18:23.660000+00:00,2022-10-17,12,0.000000,0.000,Afternoon (12-17),Q1,False
1,2022-10-17 12:18:25.472000+00:00,2022-10-17,12,0.030200,1.812,Afternoon (12-17),Q1,False
2,2022-10-17 12:18:27.348000+00:00,2022-10-17,12,0.061467,1.876,Afternoon (12-17),Q1,False
3,2022-10-17 12:18:29.222000+00:00,2022-10-17,12,0.092700,1.874,Afternoon (12-17),Q1,False
4,2022-10-17 12:18:31.117000+00:00,2022-10-17,12,0.124283,1.895,Afternoon (12-17),Q1,False


### Portions, filters and plot helpers
Portions are listed in the order they were recorded. The filters are portion, hour range, which readings to keep and an optional time average (the mean of the readings inside each 1, 5 or 15 minute window, which pulls the shape much closer to a bell). The helper draws the histogram, the fitted normal curve, the median and the chosen percentile lines. Log scale uses log(1 + x) and keeps tick labels in the original units.

In [6]:
INK, MUTED, GRID, SURFACE = '#0b0b0b', '#52514e', '#e6e5e1', '#fcfcfb'
SERIES = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100']

plt.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'axes.edgecolor': GRID,
    'axes.labelcolor': MUTED, 'xtick.color': MUTED, 'ytick.color': MUTED, 'text.color': INK,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.6, 'axes.axisbelow': True
})

portion_options = {'All data': None}
for col in ['Day Part', 'Elapsed Quarter', 'Date']:
    for val in pd.unique(robot_df[col]):
        portion_options[f'{col}: {val}'] = (col, val)


def portion_rows(portion, hours):
    d = robot_df
    key = portion_options[portion]
    if key:
        d = d[d[key[0]] == key[1]]
    return d[d['Hour'].between(*hours)]


window_options = {'Each reading': None, '1 min average': 1, '5 min average': 5, '15 min average': 15}


def pick_values(d, axis, readings, window):
    if readings == 'Non-zero readings':
        d = d[d[axis] > 0]
    elif readings == 'Active rows':
        d = d[d['Active Row']]
    minutes = window_options[window]
    if minutes:
        return d.groupby(d['Elapsed Min'] // minutes)[axis].mean().to_numpy()
    return d[axis].to_numpy()


def log_ticks(ax):
    lo, hi = ax.get_xlim()
    nice = np.array([0, 0.2, 0.5, 1, 2, 5, 10, 20, 50, 100, 200])
    pos = np.log1p(nice)
    keep = (pos >= lo) & (pos <= hi)
    ax.set_xticks(pos[keep])
    ax.set_xticklabels([f'{v:g}' for v in nice[keep]])


def draw_nd(ax, values, hist_color, curve_color, scale, pcts, labels=True):
    log = scale == 'Log'
    x = np.log1p(values) if log else values
    mu, sd = x.mean(), x.std()

    ax.hist(x, bins=50, density=True, color=hist_color, alpha=0.35, linewidth=0)
    grid = np.linspace(min(x.min(), mu - 3.5 * sd), max(x.max(), mu + 3.5 * sd), 500)
    ax.plot(grid, norm_pdf(grid, mu, sd), color=curve_color, lw=2)

    levels = sorted(set(pcts) | {50})
    q = np.percentile(values, levels)
    qx = np.log1p(q) if log else q
    if 25 in levels and 75 in levels:
        ax.axvspan(qx[levels.index(25)], qx[levels.index(75)], color=MUTED, alpha=0.08, linewidth=0)

    top = ax.get_ylim()[1] * 1.15
    ax.set_ylim(0, top)
    for i, (lv, val, pos) in enumerate(zip(levels, q, qx)):
        is_med = lv == 50
        ax.axvline(pos, color=INK if is_med else MUTED, lw=1.8 if is_med else 1, ls='-' if is_med else '--')
        if labels:
            name = 'Median' if is_med else 'P' + str(lv)
            ax.text(pos, top * (0.98 - 0.065 * (i % 3)), f'{name} {val:.2f}', fontsize=8, color=INK,
                    ha='left', va='top', bbox=dict(facecolor=SURFACE, edgecolor='none', alpha=0.8, pad=1.5))
    if log:
        log_ticks(ax)

    return dict(n=len(values), mean=values.mean(), sd=values.std(),
                median=float(np.median(values)), skew=float(np.mean(((values - values.mean()) / values.std()) ** 3)))


def norm_pdf(x, mu, sd):
    return np.exp(-0.5 * ((x - mu) / sd) ** 2) / (sd * np.sqrt(2 * np.pi))


def legend_items(hist_color, curve_color):
    return [
        Patch(color=hist_color, alpha=0.35, label='Readings'),
        Line2D([], [], color=curve_color, lw=2, label='Normal fit'),
        Line2D([], [], color=INK, lw=1.8, label='Median'),
        Line2D([], [], color=MUTED, lw=1, ls='--', label='Percentiles'),
        Patch(color=MUTED, alpha=0.15, label='P25 to P75'),
    ]


def empty_plot(ax):
    ax.text(0.5, 0.5, 'Not enough readings for this filter', ha='center', va='center', transform=ax.transAxes)
    ax.set_axis_off()

### Filter panel
Each dashboard gets its own panel. **Apply filter** redraws the plot with the current choices and **Reset** goes back to the defaults.

In [7]:
def build_controls(with_axis=False, with_portion=True, with_compare=False):
    c = {}
    if with_axis:
        c['axis'] = widgets.Dropdown(options=axes_cols, value='Axis #2', description='Axis')
    if with_portion:
        c['portion'] = widgets.Dropdown(options=list(portion_options), description='Portion')
    if with_compare:
        c['by'] = widgets.Dropdown(options=['Day Part', 'Elapsed Quarter', 'Date'], description='Compare by')
    c['readings'] = widgets.ToggleButtons(options=['Non-zero readings', 'Active rows', 'All readings'], description='Readings')
    c['window'] = widgets.Dropdown(options=list(window_options), description='Average')
    c['scale'] = widgets.ToggleButtons(options=['Raw', 'Log'], description='Scale')
    c['hours'] = widgets.IntRangeSlider(value=[0, 23], min=0, max=23, description='Hours (UTC)')
    c['pcts'] = widgets.SelectMultiple(options=[1, 5, 10, 25, 75, 90, 95, 99], value=(5, 25, 75, 95), description='Percentiles', rows=4)
    c['apply'] = widgets.Button(description='Apply filter', button_style='primary')
    c['reset'] = widgets.Button(description='Reset filters')
    return c


def launch(controls, render):
    out = widgets.Output()
    names = [k for k in controls if k not in ('apply', 'reset')]
    defaults = {k: controls[k].value for k in names}

    def run(_=None):
        with out:
            clear_output(wait=True)
            render(**{k: controls[k].value for k in names})

    def reset(_):
        for k, v in defaults.items():
            controls[k].value = v
        run()

    controls['apply'].on_click(run)
    controls['reset'].on_click(reset)
    display(widgets.VBox([controls[k] for k in names] + [widgets.HBox([controls['apply'], controls['reset']]), out]))
    run()

### Dashboard 1: one axis
Histogram of the filtered readings with the fitted normal curve. Raw readings are lumpy because the robot holds the same current for stretches, so try Log scale or a time average if the bell is not showing. The median is the solid line, the selected percentiles are dashed, and the shaded band is P25 to P75. The line under the title shows the zero share of the portion before zeros are dropped.

In [8]:
def render_single(axis, portion, readings, window, scale, hours, pcts):
    d = portion_rows(portion, hours)
    vals = pick_values(d, axis, readings, window)
    fig, ax = plt.subplots(figsize=(10, 5.4))
    if len(vals) < 15:
        empty_plot(ax)
        plt.show()
        return
    info = draw_nd(ax, vals, SERIES[0], SERIES[1], scale, pcts)
    zero_pct = (d[axis] == 0).mean() * 100
    ax.set_title(f'{axis} | {portion}', loc='left', fontsize=12, fontweight='bold', pad=24)
    ax.text(0, 1.03, f"n = {info['n']:,}   mean {info['mean']:.2f}   sd {info['sd']:.2f}   skew {info['skew']:.2f}   zeros in portion {zero_pct:.0f}%",
            transform=ax.transAxes, fontsize=9, color=MUTED)
    ax.set_xlabel('Current' + (' (log scale)' if scale == 'Log' else ''))
    ax.set_ylabel('Density')
    ax.legend(handles=legend_items(SERIES[0], SERIES[1]), frameon=False, loc='upper left',
              bbox_to_anchor=(0, -0.14), ncol=5, fontsize=9)
    plt.tight_layout()
    plt.show()


launch(build_controls(with_axis=True), render_single)

### Dashboard 2: all 8 axes
The same view for every axis side by side, using one filter panel. The median is in each panel title and the lines mark the chosen percentiles.

In [9]:
def render_grid(portion, readings, window, scale, hours, pcts):
    d = portion_rows(portion, hours)
    fig, axs = plt.subplots(2, 4, figsize=(16, 7.5))
    for ax, col in zip(axs.ravel(), axes_cols):
        vals = pick_values(d, col, readings, window)
        if len(vals) < 15:
            empty_plot(ax)
            ax.set_title(col, loc='left', fontsize=10)
            continue
        info = draw_nd(ax, vals, SERIES[0], SERIES[1], scale, pcts, labels=False)
        ax.set_title(f"{col}   median {info['median']:.2f}   n = {info['n']:,}", loc='left', fontsize=10)
    fig.suptitle(f'All axes | {portion}', x=0.01, ha='left', fontsize=13, fontweight='bold')
    fig.legend(handles=legend_items(SERIES[0], SERIES[1]), frameon=False, loc='lower center', ncol=5, fontsize=9)
    fig.subplots_adjust(left=0.04, right=0.99, top=0.88, bottom=0.12, wspace=0.22, hspace=0.35)
    plt.show()


launch(build_controls(), render_grid)

### Dashboard 3: compare portions
One axis, with a normal curve for each portion of the recording on the same chart. The dashed line in each colour is that portion's median. The table underneath gives the percentiles for each portion, so a shift between portions shows in both the chart and the numbers.

In [10]:
def render_compare(axis, by, readings, window, scale, hours, pcts):
    log = scale == 'Log'
    levels = sorted(set(pcts) | {50})
    groups = pd.unique(robot_df[by])[:4]

    series = []
    for g, color in zip(groups, SERIES):
        d = robot_df[(robot_df[by] == g) & robot_df['Hour'].between(*hours)]
        vals = pick_values(d, axis, readings, window)
        if len(vals) >= 15:
            series.append((g, color, vals, np.log1p(vals) if log else vals))

    fig, ax = plt.subplots(figsize=(10, 5.4))
    if not series:
        empty_plot(ax)
        plt.show()
        return

    left = min(min(x.min(), x.mean() - 3 * x.std()) for _, _, _, x in series)
    right = max(np.percentile(x, 99.5) for _, _, _, x in series)
    grid = np.linspace(left, right, 500)

    rows, medians = [], []
    for g, color, vals, x in series:
        pdf = norm_pdf(grid, x.mean(), x.std())
        ax.plot(grid, pdf, color=color, lw=2, label=f'{g} (n = {len(vals):,})')
        ax.fill_between(grid, pdf, color=color, alpha=0.12, linewidth=0)
        med = np.median(vals)
        med_x = np.log1p(med) if log else med
        ax.axvline(med_x, color=color, ls='--', lw=1.2)
        medians.append((g, med, med_x))
        row = {'Portion': g, 'n': len(vals), 'mean': vals.mean(), 'sd': vals.std()}
        for lv, val in zip(levels, np.percentile(vals, levels)):
            row['Median' if lv == 50 else f'P{lv}'] = val
        rows.append(row)

    ax.set_xlim(left, right)
    top = ax.get_ylim()[1] * 1.15
    ax.set_ylim(0, top)
    for i, (g, med, med_x) in enumerate(medians):
        ax.text(med_x, top * (0.98 - 0.07 * i), f"{g.split(' (')[0]} median {med:.2f}", ha='left', va='top', fontsize=8,
                color=INK, bbox=dict(facecolor=SURFACE, edgecolor='none', alpha=0.8, pad=1.5))
    ax.set_title(f'{axis} | by {by}', loc='left', fontsize=12, fontweight='bold')
    ax.set_xlabel('Current' + (' (log scale)' if log else ''))
    ax.set_ylabel('Density')
    if log:
        log_ticks(ax)
    ax.legend(frameon=False, loc='upper left', bbox_to_anchor=(0, -0.14), ncol=4, fontsize=9)
    plt.tight_layout()
    plt.show()
    display(pd.DataFrame(rows).set_index('Portion').round(2))


launch(build_controls(with_axis=True, with_portion=False, with_compare=True), render_compare)